![dvd_image](dvd_image.jpg)

A DVD rental company needs your help! They want to figure out how many days a customer will rent a DVD for based on some features and has approached you for help. They want you to try out some regression models which will help predict the number of days a customer will rent a DVD for. The company wants a model which yeilds a MSE of 3 or less on a test set. The model you make will help the company become more efficient inventory planning.

The data they provided is in the csv file `rental_info.csv`. It has the following features:
- `"rental_date"`: The date (and time) the customer rents the DVD.
- `"return_date"`: The date (and time) the customer returns the DVD.
- `"amount"`: The amount paid by the customer for renting the DVD.
- `"amount_2"`: The square of `"amount"`.
- `"rental_rate"`: The rate at which the DVD is rented for.
- `"rental_rate_2"`: The square of `"rental_rate"`.
- `"release_year"`: The year the movie being rented was released.
- `"length"`: Lenght of the movie being rented, in minuites.
- `"length_2"`: The square of `"length"`.
- `"replacement_cost"`: The amount it will cost the company to replace the DVD.
- `"special_features"`: Any special features, for example trailers/deleted scenes that the DVD also has.
- `"NC-17"`, `"PG"`, `"PG-13"`, `"R"`: These columns are dummy variables of the rating of the movie. It takes the value 1 if the move is rated as the column name and 0 otherwise. For your convinience, the reference dummy has already been dropped.

# Import the dataset

In [62]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error as MSE
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Import any additional modules and start coding below
dvd_df = pd.read_csv("rental_info.csv")
dvd_df.head()

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2
0,2005-05-25 02:54:33+00:00,2005-05-28 23:40:33+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
1,2005-06-15 23:19:16+00:00,2005-06-18 19:24:16+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
2,2005-07-10 04:27:45+00:00,2005-07-17 10:11:45+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
3,2005-07-31 12:06:41+00:00,2005-08-02 14:30:41+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401
4,2005-08-19 12:30:04+00:00,2005-08-23 13:35:04+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401


In [63]:
dvd_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15861 entries, 0 to 15860
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   rental_date       15861 non-null  object 
 1   return_date       15861 non-null  object 
 2   amount            15861 non-null  float64
 3   release_year      15861 non-null  float64
 4   rental_rate       15861 non-null  float64
 5   length            15861 non-null  float64
 6   replacement_cost  15861 non-null  float64
 7   special_features  15861 non-null  object 
 8   NC-17             15861 non-null  int64  
 9   PG                15861 non-null  int64  
 10  PG-13             15861 non-null  int64  
 11  R                 15861 non-null  int64  
 12  amount_2          15861 non-null  float64
 13  length_2          15861 non-null  float64
 14  rental_rate_2     15861 non-null  float64
dtypes: float64(8), int64(4), object(3)
memory usage: 1.8+ MB


# Get the number of rental days

In [64]:
# Convert rental_date and return_date to datetime
dvd_df["rental_date"] = pd.to_datetime(dvd_df["rental_date"])
dvd_df["return_date"] = pd.to_datetime(dvd_df["return_date"])

# Calculate rental_length
dvd_df["rental_length"] = dvd_df["return_date"] - dvd_df["rental_date"]
dvd_df["rental_length_days"] = dvd_df["rental_length"].dt.days

dvd_df.head()

,rental_date,return_date,amount,release_year,rental_rate,length,replacement_cost,special_features,NC-17,PG,PG-13,R,amount_2,length_2,rental_rate_2,rental_length,rental_length_days
0,2005-05-25 02:54:33+00:00,2005-05-28 23:40:33+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,3 days 20:46:00,3
1,2005-06-15 23:19:16+00:00,2005-06-18 19:24:16+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,2 days 20:05:00,2
2,2005-07-10 04:27:45+00:00,2005-07-17 10:11:45+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,7 days 05:44:00,7
3,2005-07-31 12:06:41+00:00,2005-08-02 14:30:41+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,2 days 02:24:00,2
4,2005-08-19 12:30:04+00:00,2005-08-23 13:35:04+00:00,2.99,2005.0,2.99,126.0,16.99,"{Trailers,""Behind the Scenes""}",0,0,0,1,8.9401,15876.0,8.9401,4 days 01:05:00,4


In [65]:
dvd_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15861 entries, 0 to 15860
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   rental_date         15861 non-null  datetime64[ns, UTC]
 1   return_date         15861 non-null  datetime64[ns, UTC]
 2   amount              15861 non-null  float64            
 3   release_year        15861 non-null  float64            
 4   rental_rate         15861 non-null  float64            
 5   length              15861 non-null  float64            
 6   replacement_cost    15861 non-null  float64            
 7   special_features    15861 non-null  object             
 8   NC-17               15861 non-null  int64              
 9   PG                  15861 non-null  int64              
 10  PG-13               15861 non-null  int64              
 11  R                   15861 non-null  int64              
 12  amount_2            15861 non-nu

# Add dummy variables to special_features

In [66]:
# Add deleted_scenes and behind_the_scenes columns
dvd_df["deleted_scenes"] = dvd_df["special_features"].str.contains("Deleted Scenes", regex = False).astype(int)
dvd_df["behind_the_scenes"] = dvd_df["special_features"].str.contains("Behind the Scenes", regex = False).astype(int)

# Drop special_features column
dvd_df = dvd_df.drop(columns = ["special_features"])

# Execute train-test split

In [67]:
# Create feature and target table
X = dvd_df.drop(columns = [
    "rental_length_days",
    "rental_length",
    "rental_date",
    "return_date"
])

y = dvd_df["rental_length_days"]

# Split train-test to 80:20 ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 9
)

# Evaluate some regression models

In [68]:
# List candidates
candidates = {
    "linreg": LinearRegression(),
    "ridge": Ridge(alpha = 1.0),
    "lasso": Lasso(alpha = 0.001, max_iter = 10000),
    "rf": RandomForestRegressor(n_estimators = 300, random_state = 9),
    "gbr": GradientBoostingRegressor(n_estimators = 300, random_state = 9)
}

# Set up best model params
best_model = None
best_mse = float("inf")

# Run each model and find the best model
for name, model in candidates.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = MSE(y_test, y_pred)

    if mse < best_mse:
        best_mse = mse
        best_model = model

    print(f"{name}: MSE = {mse:.4f}")

if best_mse < 3:
    print(f"Best model: {best_model}")
    print(f"MSE = {best_mse:.4f}")

linreg: MSE = 2.9417
ridge: MSE = 2.9418
lasso: MSE = 2.9420
rf: MSE = 2.0279
gbr: MSE = 2.2720
Best model: RandomForestRegressor(n_estimators=300, random_state=9)
MSE = 2.0279


## Why these models are used

- **LinearRegression / Ridge / Lasso:** strong baselines, fast, often great when features already include squared terms.
- **RandomForest / GradientBoosting:** handle non-linearities and interactions automatically; usually strong tabular performers.

**Why Random Forest is the best model**

Random Forest can naturally handle non-linear relationships and feature interactions without having to explicitly engineer them.

In this dataset, rental length likely depends on messy interactions like:
- **pricing signals** (amount, rental_rate, and their squared versions)
- **movie attributes** (length, replacement_cost, release_year)
- **ratings dummies** (PG, R, etc.)
- **special features flags** (deleted_scenes, behind_the_scenes)

A **linear model (Linear/Ridge/Lasso)** assumes something like:
`rental_length_days ≈ a·amount + b·length + c·release_year + …`
That’s a straight plane. Real customer behavior does not follow straightforwardly like that

On the other hand, **Random Forest**:
- splits the data into many rule-based regions (decision trees),
- captures “if/then” patterns like “if rental_rate is low AND length is high THEN rental tends to be longer”,
- averages many trees, which reduces overfitting.

In comparison to Gradient Boosting:
- **RF (bagging)** reduces variance by averaging many noisy trees → very robust.
- **GB (boosting)** sequentially fits residuals; if residuals contain noise, it can spend effort “explaining” noise unless heavily regularized.

Therefore, Random Forest is the best model in this case.

The requirement is met here, but for fun experiment, I will add some other model, as well as HistGradientBoostingRegressor, which is a modern GradientBoostingRegressor model

### P/s: Trying other models

In [69]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import ElasticNet, BayesianRidge, HuberRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor

more_candidates = {
    "dummy": DummyRegressor(strategy="mean"),
    "elasticnet": ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=9, max_iter=20000),
    "bayes_ridge": BayesianRidge(),
    "huber": Pipeline([("scaler", StandardScaler()), ("model", HuberRegressor())]),
    "knn": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsRegressor(n_neighbors=15))]),
    "svr": Pipeline([("scaler", StandardScaler()), ("model", SVR(C=10.0, epsilon=0.1))]),
    "extra_trees": ExtraTreesRegressor(n_estimators=500, random_state=9),
    "hist_gb": HistGradientBoostingRegressor(random_state=9),
}

# Set up best model params
test_best_model = None
test_best_mse = float("inf")

# Run each model and find the best model
for name, model in more_candidates.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = MSE(y_test, y_pred)

    if mse < test_best_mse:
        test_best_mse = mse
        test_best_model = model

    print(f"{name}: MSE = {mse:.4f}")

if test_best_mse < 3:
    print(f"Best model: {test_best_model}")
    print(f"MSE = {test_best_mse:.4f}")

dummy: MSE = 7.1000
elasticnet: MSE = 2.9421
bayes_ridge: MSE = 2.9421
huber: MSE = 2.9440
knn: MSE = 3.3449
svr: MSE = 2.2536
extra_trees: MSE = 2.0337
hist_gb: MSE = 2.0752
Best model: ExtraTreesRegressor(n_estimators=500, random_state=9)
MSE = 2.0337


Even though **HistGradientBoostingRegressor** is strong (MSE = 2.0752), **ExtraTreesRegression** has the best value, with MSE = 2.0337. However, it pales in comparison to **Random Forest** (MSE = 2.0279). Therefore, **Random Forest is the best model overall in this case**.